# Colab 一体化推理 + 全参考评估 (PSNR / SSIM / LPIPS / Edge F1)

在 Colab 上使用 A100，**一次性对 4 个模型**做推理并计算全参考指标，结果写入 Google Drive。

### zip 内需要包含的内容（解压后目录结构）

`dataset.zip` 解压到 `/content/` 后，应得到：

| 路径 | 说明 |
|------|------|
| `dataset/highres/original/` | HR 原图（必须） |
| `dataset/lowres_4x_simple/original/` | 4× 简化退化 LR（评估 4x 模型时用） |
| `dataset/lowres_2x_simple/original/` | 2× 简化退化 LR（评估 2x 模型时用） |
| `pretrained_models/2x_APISR_RRDB_GAN_generator.pth` | APISR 2x 预训练（可选，仅当评估 APISR 2x 时需要） |
| `pretrained_models/4x_APISR_RRDB_GAN_generator.pth` | APISR 4x 预训练（可选，仅当评估 APISR 4x 时需要） |

- **我们自己的 best 模型**（如 `ESRGAN_4x_23block_simple_best.pth`）放在 **Google Drive** 的 `AWM/saved_models/`，不要放进 zip；Colab 会通过软链接 `saved_models` → Drive 读取。
- 若只评估部分模型，可在下方配置列表 `MODEL_CONFIGS` 中注释掉不需要的项。

In [ ]:
# ============================================================
# 0. 全局配置 + 4 个模型列表（一次性推理并评估）
# ============================================================

GITHUB_REPO      = 'https://github.com/liqiqinaOH7/AWM.git'
BRANCH           = 'main'
ONEDRIVE_ZIP_URL = 'https://ucsdcloud-my.sharepoint.com/:u:/g/personal/xil326_ucsd_edu/IQDHy1rPvIVyQI3YDim7rLWVAfUH7sGLFDzt3-hWzE5cW04?download=1'
DRIVE_ROOT       = '/content/drive/MyDrive/AWM'
PROJECT_DIR      = '/content/AWM'

HR_DIR = 'dataset/highres/original'
RESULTS_DIR = 'results'

# 4 个模型配置（按顺序推理并评估）。不需要的可以注释掉或删掉。
CKPT_APISR_2X = 'pretrained_models/2x_APISR_RRDB_GAN_generator.pth'
CKPT_APISR_4X = 'pretrained_models/4x_APISR_RRDB_GAN_generator.pth'

MODEL_CONFIGS = [
    {'model_name': 'ESRGAN_4x_23block_simple', 'scale': 4, 'num_block': 23, 'is_apisr': False,
     'ckpt_ours': 'saved_models/ESRGAN_4x_23block_simple_best.pth', 'lr_dir': 'dataset/lowres_4x_simple/original'},
    {'model_name': 'ESRGAN_2x_23block_simple', 'scale': 2, 'num_block': 23, 'is_apisr': False,
     'ckpt_ours': 'saved_models/ESRGAN_2x_23block_simple_best.pth', 'lr_dir': 'dataset/lowres_2x_simple/original'},
    {'model_name': 'APISR_RRDB_4x', 'scale': 4, 'num_block': 23, 'is_apisr': True,
     'ckpt_ours': None, 'lr_dir': 'dataset/lowres_4x_simple/original'},
    {'model_name': 'APISR_RRDB_2x', 'scale': 2, 'num_block': 23, 'is_apisr': True,
     'ckpt_ours': None, 'lr_dir': 'dataset/lowres_2x_simple/original'},
]

print(f'本轮将依次评估 {len(MODEL_CONFIGS)} 个模型。')


In [ ]:
# ============================================================
# 1. 挂载 Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
for d in ['saved_models', 'results']:
    os.makedirs(f'{DRIVE_ROOT}/{d}', exist_ok=True)
print(f'Drive 就绪: {DRIVE_ROOT}')

In [ ]:
# ============================================================
# 2. 从 OneDrive 下载并解压 dataset.zip（含 dataset/ + pretrained_models/）
# ============================================================
import os, time

zip_path = '/content/dataset.zip'
marker   = '/content/dataset/highres/original'

if os.path.isdir(marker):
    print('dataset 已存在，跳过下载。')
else:
    print('正在从 OneDrive 下载 dataset.zip（~3.4 GB）...')
    t0 = time.time()
    !wget -q --show-progress -O "{zip_path}" "{ONEDRIVE_ZIP_URL}"
    elapsed = time.time() - t0
    if os.path.isfile(zip_path) and os.path.getsize(zip_path) > 1_000_000:
        print(f'下载完成: {os.path.getsize(zip_path)/1e9:.2f} GB, {elapsed:.0f}s，解压中...')
        !unzip -q -o "{zip_path}" -d /content/
        os.remove(zip_path)
        print('解压完成。')
    else:
        print('⚠ 下载可能失败，请检查 OneDrive 链接权限和 URL。')

In [ ]:
# ============================================================
# 2b. 备用：若 wget 失败，可手动上传 dataset.zip 后运行
# ============================================================
# from google.colab import files
# uploaded = files.upload()
# import os
# if os.path.isfile('/content/dataset.zip'):
#     !unzip -q -o /content/dataset.zip -d /content/
#     os.remove('/content/dataset.zip')
#     print('解压完成。')

In [ ]:
# ============================================================
# 3. 拉取 GitHub 代码 + 创建软链接 + 安装依赖
# ============================================================
import os

if os.path.isdir(PROJECT_DIR):
    !cd {PROJECT_DIR} && git pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {GITHUB_REPO} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print(f'工作目录: {os.getcwd()}')

links = {
    'dataset':           '/content/dataset',
    'pretrained_models': '/content/pretrained_models',
    'saved_models':      f'{DRIVE_ROOT}/saved_models',
    'results':           f'{DRIVE_ROOT}/results',
}

for name, target in links.items():
    lp = os.path.join(PROJECT_DIR, name)
    if os.path.islink(lp):
        if os.readlink(lp) == target:
            continue
        os.unlink(lp)
    elif os.path.isdir(lp):
        continue
    os.symlink(target, lp)
    print(f'  链接: {name} → {target}')

# 评估依赖：lpips / scikit-image / tqdm
!pip install -q lpips scikit-image tqdm
print('\n环境初始化完成。')

In [ ]:
# ============================================================
# 4. 导入依赖 & 构建模型
# ============================================================

import glob
import json
import os
from math import log10

import cv2
import numpy as np
from tqdm import tqdm

import torch
import torch.nn.functional as F

from skimage.metrics import structural_similarity as ssim
import lpips

from architecture import build_rrdb

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

def load_rrdb_model(scale: int, num_block: int, is_apisr: bool, ckpt_ours: str = None):
    model = build_rrdb(scale=scale, num_block=num_block).to(device)
    model.eval()
    ckpt_path = (CKPT_APISR_2X if scale == 2 else CKPT_APISR_4X) if is_apisr else ckpt_ours
    if not ckpt_path or not os.path.isfile(ckpt_path):
        raise FileNotFoundError(f'权重不存在: {ckpt_path}')
    ckpt = torch.load(ckpt_path, map_location=device)
    if 'params_ema' in ckpt: state = ckpt['params_ema']
    elif 'model_state_dict' in ckpt: state = ckpt['model_state_dict']
    else: state = ckpt
    model_state = model.state_dict()
    compatible = {k: v for k, v in state.items() if k in model_state and model_state[k].shape == v.shape}
    model.load_state_dict({**model_state, **compatible}, strict=False)
    if is_apisr: print(f'  加载 APISR: {ckpt_path}, 兼容 {len(compatible)}/{len(model_state)}')
    else: print(f'  加载自训练: {ckpt_path}, Epoch {ckpt.get("epoch", "?")}')

    return model


In [ ]:
# ============================================================
# 5. 推理函数（支持 2x/4x + 大图分块）
# ============================================================

def sr_infer_single(model, lr_img_bgr: np.ndarray, scale: int) -> np.ndarray:
    """对单张 LR 图像做超分，返回 RGB uint8 图像。"""
    lr_img = cv2.cvtColor(lr_img_bgr, cv2.COLOR_BGR2RGB)
    h, w = lr_img.shape[:2]

    pad_h = (4 - h % 4) % 4 if scale == 2 else 0
    pad_w = (4 - w % 4) % 4 if scale == 2 else 0
    lr_padded = lr_img
    if pad_h > 0 or pad_w > 0:
        lr_padded = cv2.copyMakeBorder(lr_img, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT_101)

    lr_t = torch.from_numpy(lr_padded.transpose(2, 0, 1)).float().unsqueeze(0).to(device) / 255.0

    with torch.no_grad():
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            _, _, hh, ww = lr_t.shape
            if hh * ww > 512 * 512:
                h_half, w_half = hh // 2, ww // 2
                sr_t = torch.zeros((1, 3, hh * scale, ww * scale), device=device)
                sr_t[:, :, :h_half*scale, :w_half*scale] = model(lr_t[:, :, :h_half, :w_half])
                sr_t[:, :, :h_half*scale, w_half*scale:] = model(lr_t[:, :, :h_half, w_half:])
                sr_t[:, :, h_half*scale:, :w_half*scale] = model(lr_t[:, :, h_half:, :w_half])
                sr_t[:, :, h_half*scale:, w_half*scale:] = model(lr_t[:, :, h_half:, w_half:])
            else:
                sr_t = model(lr_t)

    sr = sr_t.squeeze(0).float().cpu().clamp(0, 1).numpy().transpose(1, 2, 0)
    sr = sr[:h * scale, :w * scale, :]
    sr_uint8 = (sr * 255.0).round().astype(np.uint8)
    return sr_uint8


def ensure_dir(path: str):
    if path and not os.path.isdir(path):
        os.makedirs(path, exist_ok=True)


In [ ]:
# ============================================================
# 6. 全参考指标：PSNR / SSIM / LPIPS / Canny Edge F1（按 MODEL_CONFIGS 循环）
# ============================================================

lpips_model = lpips.LPIPS(net='vgg').to(device)
lpips_model.eval()

# DISTS 指标（来自 piq）
import piq
try:
    dists_model = piq.DISTS().to(device)
    dists_model.eval()
except Exception as e:
    print('DISTS 初始化失败，请确认已安装 piq:', e)
    dists_model = None

def img_to_tensor_lpips(img_rgb: np.ndarray) -> torch.Tensor:
    x = img_rgb.astype(np.float32) / 255.0
    x = x * 2.0 - 1.0
    x = torch.from_numpy(x.transpose(2, 0, 1)).unsqueeze(0)
    return x.to(device)


def img_to_tensor_dists(img_rgb: np.ndarray) -> torch.Tensor:
    """DISTS 使用 [0,1] 归一化张量。"""
    x = img_rgb.astype(np.float32) / 255.0
    x = torch.from_numpy(x.transpose(2, 0, 1)).unsqueeze(0)
    return x.to(device)


def compute_lpips_patchwise(hr_rgb: np.ndarray, sr_rgb: np.ndarray, patch_size: int = 256, batch_size: int = 4) -> float:
    """将整图切成 patch，分 batch 计算 LPIPS，最后取平均。"""
    h, w = hr_rgb.shape[:2]
    ps = patch_size
    lpips_vals = []

    for y in range(0, h, ps):
        for x in range(0, w, ps):
            hr_patch = hr_rgb[y:y+ps, x:x+ps, :]
            sr_patch = sr_rgb[y:y+ps, x:x+ps, :]
            if hr_patch.size == 0 or sr_patch.size == 0:
                continue
            hr_t = img_to_tensor_lpips(hr_patch)
            sr_t = img_to_tensor_lpips(sr_patch)
            lpips_vals.append((hr_t, sr_t))

    scores = []
    with torch.no_grad():
        for i in range(0, len(lpips_vals), batch_size):
            batch = lpips_vals[i:i+batch_size]
            hr_batch = torch.cat([h for h, _ in batch], dim=0)
            sr_batch = torch.cat([s for _, s in batch], dim=0)
            val = lpips_model(hr_batch, sr_batch)
            scores.extend(val.view(-1).cpu().tolist())

    return float(np.mean(scores)) if scores else float('nan')


def compute_dists_patchwise(hr_rgb: np.ndarray, sr_rgb: np.ndarray, patch_size: int = 256, batch_size: int = 4) -> float:
    """将整图切成 patch，分 batch 计算 DISTS，最后取平均。"""
    if dists_model is None:
        return float('nan')

    h, w = hr_rgb.shape[:2]
    ps = patch_size
    pairs = []

    for y in range(0, h, ps):
        for x in range(0, w, ps):
            hr_patch = hr_rgb[y:y+ps, x:x+ps, :]
            sr_patch = sr_rgb[y:y+ps, x:x+ps, :]
            if hr_patch.size == 0 or sr_patch.size == 0:
                continue
            hr_t = img_to_tensor_dists(hr_patch)
            sr_t = img_to_tensor_dists(sr_patch)
            pairs.append((hr_t, sr_t))

    scores = []
    with torch.no_grad():
        for i in range(0, len(pairs), batch_size):
            batch = pairs[i:i+batch_size]
            hr_batch = torch.cat([h for h, _ in batch], dim=0)
            sr_batch = torch.cat([s for _, s in batch], dim=0)
            val = dists_model(hr_batch, sr_batch)
            scores.extend(val.view(-1).cpu().tolist())

    return float(np.mean(scores)) if scores else float('nan')

def canny_f1(sr_gray: np.ndarray, hr_gray: np.ndarray, t1: int = 100, t2: int = 200) -> float:
    sr_edges = cv2.Canny(sr_gray, t1, t2)
    hr_edges = cv2.Canny(hr_gray, t1, t2)
    sr_bin = sr_edges > 0
    hr_bin = hr_edges > 0
    tp = np.logical_and(sr_bin, hr_bin).sum()
    fp = np.logical_and(sr_bin, ~hr_bin).sum()
    fn = np.logical_and(~sr_bin, hr_bin).sum()
    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    return float(f1)

hr_dir_abs = os.path.join(PROJECT_DIR, HR_DIR)
all_summaries = []

for cfg in MODEL_CONFIGS:
    model_name = cfg['model_name']
    scale = cfg['scale']
    num_block = cfg['num_block']
    is_apisr = cfg['is_apisr']
    lr_dir = os.path.join(PROJECT_DIR, cfg['lr_dir'])
    sr_dir = os.path.join(RESULTS_DIR, f"{model_name}_fullref_sr")
    ensure_dir(sr_dir)
    ensure_dir(RESULTS_DIR)

    model = load_rrdb_model(scale, num_block, is_apisr, cfg.get('ckpt_ours'))
    lr_paths = sorted(glob.glob(os.path.join(lr_dir, '*.[jp][pn]*g')))
    print(f'[{model_name}] LR 图像数: {len(lr_paths)}')

    psnr_list, ssim_list, lpips_list, f1_list, dists_list = [], [], [], [], []
    per_image_scores = {}

    for lr_path in tqdm(lr_paths, desc=model_name):
        fname = os.path.basename(lr_path)
        stem, _ext = os.path.splitext(fname)
        hr_path = None
        for ext in ['.png', '.jpg', '.jpeg', '.JPG', '.JPEG']:
            cand = os.path.join(hr_dir_abs, stem + ext)
            if os.path.isfile(cand):
                hr_path = cand
                break
        if hr_path is None:
            continue

        hr_bgr = cv2.imread(hr_path)
        if hr_bgr is None:
            continue
        hr_rgb = cv2.cvtColor(hr_bgr, cv2.COLOR_BGR2RGB)

        lr_bgr = cv2.imread(lr_path)
        if lr_bgr is None:
            continue
        sr_rgb = sr_infer_single(model, lr_bgr, scale)
        cv2.imwrite(os.path.join(sr_dir, fname), cv2.cvtColor(sr_rgb, cv2.COLOR_RGB2BGR))

        h_hr, w_hr = hr_rgb.shape[:2]
        h_sr, w_sr = sr_rgb.shape[:2]
        if (h_hr != h_sr) or (w_hr != w_sr):
            sr_rgb = cv2.resize(sr_rgb, (w_hr, h_hr), interpolation=cv2.INTER_CUBIC)

        hr_f = hr_rgb.astype(np.float32) / 255.0
        sr_f = sr_rgb.astype(np.float32) / 255.0

        mse = np.mean((hr_f - sr_f) ** 2)
        psnr_val = float('inf') if mse == 0 else (10 * log10(1.0 / mse))
        ssim_val = ssim(hr_f, sr_f, channel_axis=2, data_range=1.0)

        # LPIPS / DISTS 使用 patch + batch 方式计算，避免整图占用显存
        lpips_val = compute_lpips_patchwise(hr_rgb, sr_rgb, patch_size=256, batch_size=4)
        dists_val = compute_dists_patchwise(hr_rgb, sr_rgb, patch_size=256, batch_size=4)

        hr_gray = cv2.cvtColor(hr_rgb, cv2.COLOR_RGB2GRAY)
        sr_gray = cv2.cvtColor(sr_rgb, cv2.COLOR_RGB2GRAY)
        f1_val = canny_f1(sr_gray, hr_gray)

        psnr_list.append(psnr_val)
        ssim_list.append(ssim_val)
        lpips_list.append(lpips_val)
        dists_list.append(dists_val)
        f1_list.append(f1_val)
        per_image_scores[fname] = {
            'PSNR': round(psnr_val, 4),
            'SSIM': round(ssim_val, 4),
            'LPIPS': round(lpips_val, 4),
            'DISTS': round(dists_val, 4) if not np.isnan(dists_val) else None,
            'EdgeF1': round(f1_val, 4),
        }

    n = len(psnr_list)
    print(f'[{model_name}] 有效样本: {n}')
    if n == 0:
        del model
        torch.cuda.empty_cache()
        continue

    avg_psnr = float(np.mean(psnr_list))
    avg_ssim = float(np.mean(ssim_list))
    avg_lpips = float(np.mean(lpips_list))
    avg_dists = float(np.mean(dists_list)) if dists_list else float('nan')
    avg_f1 = float(np.mean(f1_list))
    all_summaries.append({
        'model_name': model_name,
        'PSNR': avg_psnr,
        'SSIM': avg_ssim,
        'LPIPS': avg_lpips,
        'DISTS': avg_dists,
        'EdgeF1': avg_f1,
        'num_images': n,
    })

    summary = {
        'model': model_name,
        'scale': scale,
        'num_block': num_block,
        'is_apisr': is_apisr,
        'num_images': n,
        'metrics': {
            'PSNR': round(avg_psnr, 4),
            'SSIM': round(avg_ssim, 4),
            'LPIPS': round(avg_lpips, 4),
            'DISTS': round(avg_dists, 4) if not np.isnan(avg_dists) else None,
            'EdgeF1': round(avg_f1, 4),
        },
        'per_image': per_image_scores,
    }
    json_path = os.path.join(RESULTS_DIR, f'{model_name}_fullref_eval.json')
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    txt_path = os.path.join(RESULTS_DIR, f'{model_name}_fullref_eval.txt')
    with open(txt_path, 'w', encoding='utf-8') as f:
        f.write(f'{model_name} (scale={scale}x, blocks={num_block}, is_apisr={is_apisr})\n')
        f.write(f'num_images = {n}\n' + '='*40 + '\n')
        f.write(f'PSNR   (↑, dB)     : {avg_psnr:.4f}\n')
        f.write(f'SSIM   (↑)         : {avg_ssim:.4f}\n')
        f.write(f'LPIPS  (↓, VGG)    : {avg_lpips:.4f}\n')
        f.write(f'Edge F1 (↑, Canny) : {avg_f1:.4f}\n' + '='*40 + '\n')
    print(f'  已保存: {json_path}, {txt_path}')

    del model
    torch.cuda.empty_cache()

print('所有模型评估完成。')


In [ ]:
# ============================================================
# 7. 汇总表：4 个模型对比（PSNR / SSIM / LPIPS / Edge F1）
# ============================================================

if not all_summaries:
    print('⚠ 无有效评估结果，请先运行上一 cell 并检查 LR_DIR / HR_DIR 与 MODEL_CONFIGS。')
else:
    print('\n' + '='*80)
    print('  全参考评估汇总（↑ 越高越好：PSNR / SSIM / Edge F1；↓ 越低越好：LPIPS）')
    print('='*80)
    fmt = '{:<30} {:>9} {:>9} {:>9} {:>9} {:>9}'
    print(fmt.format('Model', 'PSNR↑', 'SSIM↑', 'LPIPS↓', 'DISTS↓', 'EdgeF1↑'))
    print('-'*80)
    for s in all_summaries:
        print(fmt.format(
            s['model_name'][:29],
            f"{s['PSNR']:.4f}",
            f"{s['SSIM']:.4f}",
            f"{s['LPIPS']:.4f}",
            f"{s['DISTS']:.4f}" if not np.isnan(s['DISTS']) else '  nan',
            f"{s['EdgeF1']:.4f}",
        ))
    print('='*80)
    print('✅ 各模型 JSON/TXT 已保存至 RESULTS_DIR，汇总表如上。')
